# make-meme: free remote GPU backend (Kaggle / Colab)

Runs ComfyUI with the make-meme recipes on a free cloud GPU and gives you a URL to put in
`COMFY_URL` for your local make-meme MCP server.

* **Kaggle** (recommended, ~30 GPU h/week free): *Settings → Accelerator: GPU T4 x2*, *Internet: on*.
* **Colab**: *Runtime → Change runtime type → T4 GPU*.

⚠️ The tunnel URL is public: anyone who knows it can use this ComfyUI while the notebook runs.
Don't share it, and stop the notebook when you're done.

In [ ]:
import os, sys, subprocess, pathlib
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content")
COMFY_DIR = WORK / "ComfyUI"
os.environ["MAKE_MEME_HOME"] = str(WORK / "make-meme")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip())

In [ ]:
# 1) install make-meme (CLI only; the MCP server itself runs on your computer)
!pip -q install "make-meme-mcp @ git+https://github.com/kwanhyeonpark-ctrl/make-meme.git"
# 2) ComfyUI + custom nodes + models (~25 GB download, 10-20 min the first time).
#    --system-python reuses the notebook's preinstalled CUDA PyTorch.
!make-meme install --comfy-dir {COMFY_DIR} --system-python

In [ ]:
# 3) start ComfyUI in the background
log = open(WORK / "comfyui.log", "w")
comfy = subprocess.Popen([str(COMFY_DIR / ".venv/bin/python"), "main.py", "--listen", "127.0.0.1", "--port", "8188",
                          "--disable-auto-launch"], cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT)
import time, urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); print("ComfyUI is up"); break
    except Exception:
        time.sleep(5)
else:
    print(open(WORK / "comfyui.log").read()[-3000:])

In [ ]:
# 4) public tunnel (Cloudflare quick tunnel, no account needed)
import re
if not (WORK / "cloudflared").exists():
    !wget -q -O {WORK}/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {WORK}/cloudflared
tunnel = subprocess.Popen([str(WORK / "cloudflared"), "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("\nCOMFY_URL =", url)
print("\nClaude Code:  claude mcp add make-meme -e COMFY_URL=" + str(url) + " -- uvx make-meme-mcp")

In [ ]:
# 5) keep the session alive and show ComfyUI's latest log lines (stop this cell to shut down)
try:
    while comfy.poll() is None:
        time.sleep(60)
        print(open(WORK / "comfyui.log").read().splitlines()[-1:])
finally:
    comfy.terminate(); tunnel.terminate()